In [3]:
import numpy as np

dim = int(input("Enter number of intersections: "))
pop_size = int(input("Enter number of grey wolves (population size): "))
max_iter = int(input("Enter maximum iterations: "))
cycle_length = float(input("Enter total signal cycle length in seconds: "))
min_green = float(input("Enter minimum green time in seconds: "))
max_green = float(input("Enter maximum green time in seconds: "))

arrival_rates = np.array(
    list(
        map(
            float,
            input(
                f"Enter {dim} vehicle arrival rates (veh/sec) separated by"
                " space: "
            ).split(),
        )
    )
)
saturation_flow = np.array(
    list(
        map(
            float,
            input(
                f"Enter {dim} saturation flow rates (veh/sec) separated by"
                " space: "
            ).split(),
        )
    )
)
initial_queues = np.array(
    list(
        map(
            float,
            input(
                f"Enter {dim} initial queue lengths (vehicles) separated by"
                " space: "
            ).split(),
        )
    )
)


def objective_function(green_times):
    total_wait = 0.0
    total_queue = 0.0


    lost_time_per_phase = 3.0
    available_green = cycle_length - (dim * lost_time_per_phase)

    for i in range(len(green_times)):
        g = green_times[i]
        lam = arrival_rates[i]
        mu = saturation_flow[i]
        q0 = initial_queues[i]

        residual_queue = max(0.0, q0 + (lam * cycle_length) - (mu * g))
        green_ratio = g / cycle_length
        avg_delay = (0.5 * cycle_length * (1 - green_ratio) ** 2) / max(
            0.1, (1 - (lam / mu))
        )

        total_wait += avg_delay
        total_queue += residual_queue


    cycle_violation = abs(sum(green_times) - available_green)
    penalty = 100.0 * cycle_violation

    return 0.6 * total_wait + 0.4 * total_queue + penalty



def gwo_traffic():
  positions = np.random.uniform(min_green, max_green, (pop_size, dim))
  alpha_pos = np.zeros(dim)
  alpha_score = float("inf")
  beta_pos = np.zeros(dim)
  beta_score = float("inf")
  delta_pos = np.zeros(dim)
  delta_score = float("inf")

  for t in range(max_iter):
    for i in range(pop_size):
      positions[i] = np.clip(positions[i], min_green, max_green)
      fitness = objective_function(positions[i])

      if fitness < alpha_score:
        delta_score, delta_pos = beta_score, beta_pos.copy()
        beta_score, beta_pos = alpha_score, alpha_pos.copy()
        alpha_score, alpha_pos = fitness, positions[i].copy()
      elif fitness < beta_score:
        delta_score, delta_pos = beta_score, beta_pos.copy()
        beta_score, beta_pos = fitness, positions[i].copy()
      elif fitness < delta_score:
        delta_score, delta_pos = fitness, positions[i].copy()

    a = 2.0 - t * (2.0 / max_iter)

    for i in range(pop_size):
      for j in range(dim):
        r1, r2 = np.random.random(), np.random.random()
        A1 = 2 * a * r1 - a
        C1 = 2 * r2
        D_alpha = abs(C1 * alpha_pos[j] - positions[i][j])
        X1 = alpha_pos[j] - A1 * D_alpha

        r1, r2 = np.random.random(), np.random.random()
        A2 = 2 * a * r1 - a
        C2 = 2 * r2
        D_beta = abs(C2 * beta_pos[j] - positions[i][j])
        X2 = beta_pos[j] - A2 * D_beta

        r1, r2 = np.random.random(), np.random.random()
        A3 = 2 * a * r1 - a
        C3 = 2 * r2
        D_delta = abs(C3 * delta_pos[j] - positions[i][j])
        X3 = delta_pos[j] - A3 * D_delta

        positions[i][j] = (X1 + X2 + X3) / 3.0

  return alpha_pos, alpha_score


best_timings, best_cost = gwo_traffic()

print("\n--- Optimal Signal Timings ---")
for idx, g_time in enumerate(best_timings, start=1):
  print(f"Intersection {idx} Green Time: {g_time:.2f} sec")
print(f"Minimized Objective Cost: {best_cost:.2f}")


Enter number of intersections: 4
Enter number of grey wolves (population size): 25
Enter maximum iterations: 50
Enter total signal cycle length in seconds: 100
Enter minimum green time in seconds: 15
Enter maximum green time in seconds: 75
Enter 4 vehicle arrival rates (veh/sec) separated by space: 0.45 0.60 0.35 0.55
Enter 4 saturation flow rates (veh/sec) separated by space: 1.20 1.30 1.10 1.25 
Enter 4 initial queue lengths (vehicles) separated by space: 18 25 12 22

--- Optimal Signal Timings ---
Intersection 1 Green Time: 16.85 sec
Intersection 2 Green Time: 18.36 sec
Intersection 3 Green Time: 27.59 sec
Intersection 4 Green Time: 25.19 sec
Minimized Objective Cost: 189.90
